In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
import gc
import matplotlib.pyplot as plt
import shap
from adjustText import adjust_text
import os
import time

C:\Users\Jonas\miniconda3\envs\mlenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==============================================================================
# 1. LOAD AND MERGE ALL DATASETS (Strict Row-Count Control)
# ==============================================================================
print("Loading VAE outputs...")
vae_files = {
    'NP': ['TAS', 'hfls', 'zg', 'SIC'],
    'BK': ['TAS', 'hfls', 'zg', 'SIC'],
    'NAO': ['TAS', 'hfls', 'zg']
}

master_df = None
for region, variables in vae_files.items():
    for var in variables:
        filename = f"Data/{region}_{var}_full_latent_vaeb.csv" # delete _vaeb to go back 1 beta VAE change to ae for normal ae
        try:
            df = pd.read_csv(filename)
            df['time'] = pd.to_datetime(df['time'])
            
            float_cols = df.select_dtypes(include=['float64']).columns
            df[float_cols] = df[float_cols].astype('float32')
            
            df = df.drop_duplicates(subset=['member', 'time'])
            
            rename_map = {col: col.replace('latent_', f'latent_{region}_{var.upper()}_') 
                          for col in df.columns if col.startswith('latent_')}
            df = df.rename(columns=rename_map)
            
            if master_df is None:
                master_df = df
            else:
                master_df = pd.merge(master_df, df, on=['member', 'time'], how='left', validate='1:1')
        except FileNotFoundError:
            pass 

print("Loading Target and External files...")
ds_t = pd.read_csv('Data/europe_temp.csv')
ds_t['time'] = pd.to_datetime(ds_t['time'])
ds_target = ds_t[ds_t['time'].dt.year > 1969].copy()

ds_target = ds_target.drop_duplicates(subset=['member', 'time'])

master_df = pd.merge(
    master_df, 
    ds_target[['member', 'time', 'europe_temp']], 
    on=['member', 'time'], 
    how='inner', 
    validate='1:1'
)

master_df['month'] = master_df['time'].dt.month

bk_sic_df = pd.read_csv('Data/BK_SIC.csv')
npl_sic_df = pd.read_csv('Data/NPL_SIC.csv')
bk_sic_df['time'] = pd.to_datetime(bk_sic_df['time'])
npl_sic_df['time'] = pd.to_datetime(npl_sic_df['time'])

bk_sic_df = bk_sic_df.drop_duplicates(subset=['member', 'time'])
npl_sic_df = npl_sic_df.drop_duplicates(subset=['member', 'time'])

master_df = pd.merge(master_df, bk_sic_df[['time', 'member', 'BK_SIC']], on=['member', 'time'], how='left', validate='1:1')
master_df = pd.merge(master_df, npl_sic_df[['time', 'member', 'NPL_SIC']], on=['member', 'time'], how='left', validate='1:1')

ds_lin = pd.read_csv('Data/ds_bk_all_experiments.csv')
ds_lin['time'] = pd.to_datetime(ds_lin['time'])
ds_lin = ds_lin.drop(columns=['BK_SIC', 'europe_temp'], errors='ignore') 
ds_lin = ds_lin.drop_duplicates(subset=['member', 'time']) 

lin_float_cols = ds_lin.select_dtypes(include=['float64']).columns
ds_lin[lin_float_cols] = ds_lin[lin_float_cols].astype('float32')

master_df = pd.merge(master_df, ds_lin, on=['member', 'time'], how='left', validate='1:1')

# ==============================================================================
# 2. FILTER & EXTRACT FEATURES
# ==============================================================================
print("Filtering dataset...")
valid_months = [10, 11, 12, 1, 2, 3]
mask = master_df['time'].dt.month.isin(valid_months) & (master_df['BK_SIC'] > 1)
master_df = master_df[mask].copy()

#print(f"CRITICAL CHECK Master DataFrame rows after filtering: {len(master_df)}")

all_latent_cols = [col for col in master_df.columns if col.startswith('latent_')]
features_sic = [col for col in all_latent_cols if '_SIC_' in col]
features_tas = [col for col in all_latent_cols if '_TAS_' in col]
features_hfls = [col for col in all_latent_cols if '_HFLS_' in col]
features_zg = [col for col in all_latent_cols if '_ZG_' in col]

features_bk = [col for col in all_latent_cols if '_BK_' in col]
features_nao = [col for col in all_latent_cols if '_NAO_' in col]
features_np = [col for col in all_latent_cols if '_NP_' in col]

features_bk_sic = [col for col in all_latent_cols if '_BK_SIC_' in col]
features_bk_tas = [col for col in all_latent_cols if '_BK_TAS_' in col]
features_bk_hfls = [col for col in all_latent_cols if '_BK_HFLS_' in col]
features_bk_zg = [col for col in all_latent_cols if '_BK_ZG_' in col]

latent_cols_lin = [col for col in ds_lin.columns if col not in ['member', 'time']]
features_exp_BK = [col for col in latent_cols_lin if '(+1) BK' in col]
features_exp_EU = [col for col in latent_cols_lin if '(+1) EU' in col]
features_exp = [col for col in latent_cols_lin if col not in features_exp_BK and col not in features_exp_EU]

float_cols = master_df.select_dtypes(include=['float64']).columns
master_df[float_cols] = master_df[float_cols].astype('float32')
del ds_t, bk_sic_df, npl_sic_df, ds_lin, ds_target
gc.collect()

# ==============================================================================
# 3. CONFIGURE EXPERIMENTS
# ==============================================================================
base_cols = ['month', 'BK_SIC', 'NPL_SIC']

standard_experiments = {
    'Latent BK SIC Only': features_bk_sic + base_cols,
    'Latent BK TAS Only': features_bk_tas + base_cols,
    'Latent BK HFLS Only': features_bk_hfls + base_cols,
    'Latent BK ZG Only': features_bk_zg + base_cols,

    'Latent BK Region Only': features_bk + base_cols,
    'Latent NAO Region Only': features_nao + base_cols,
    'Latent NP Region Only': features_np + base_cols,

    'Latent BK + NAO Region Only': features_bk + features_nao + base_cols,
    'Latent NP + NAO Region Only': features_np + features_nao + base_cols,
    'Latent BK + NP Region Only': features_bk + features_np + base_cols,
    
    'Latent SIC Only': features_sic + base_cols,
    'Latent TAS Only': features_tas + base_cols,
    'Latent HFLS Only': features_hfls + base_cols,
    'Latent ZG Only': features_zg + base_cols,
    'Latent BK Region Only': features_bk + base_cols,
    'All Combined Only': all_latent_cols + base_cols,
    
    'Latent SIC + Linear Guess': features_sic + features_exp + base_cols,
    'Latent TAS + Linear Guess': features_tas + features_exp + base_cols,  
    'Latent HFLS + Linear Guess': features_hfls + features_exp + base_cols,
    'Latent ZG + Linear Guess': features_zg + features_exp + base_cols,
    'Latent BK Region + Linear Guess': features_bk + features_exp + base_cols,
    'All Combined + Linear Guess': all_latent_cols + features_exp + base_cols,
}


target_col = 'europe_temp'
val_members = ['r106', 'r117', 'r139']
valid_df = master_df[master_df['member'].isin(val_members)].copy()
train_df = master_df[~master_df['member'].isin(val_members)].copy()

# ==============================================================================
# 4. OPTUNA OBJECTIVE FUNCTION
# ==============================================================================
def create_objective(X_t, y_t, X_v, y_v, margin_t=None, margin_v=None):
    def objective(trial):
        early_stop = xgb.callback.EarlyStopping(rounds=25, min_delta=0.0005, save_best=True, maximize=False)
        
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 256, 2400),
            'learning_rate': trial.suggest_float('learning_rate', 0.00025, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 11),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 0.85, 10),
            'gamma': trial.suggest_float('gamma', 1e-8, 0.1, log=True),
            'random_state': 42,
            'tree_method': 'hist',
            'device': 'cuda',
            'objective': 'reg:squarederror', 
            'eval_metric': 'rmse',
            'callbacks': [early_stop]
        }

        model = xgb.XGBRegressor(**param)
        fit_kwargs = {'X': X_t, 'y': y_t, 'eval_set': [(X_v, y_v)], 'verbose': False}
        
        if margin_t is not None:
            fit_kwargs['base_margin'] = margin_t
            fit_kwargs['base_margin_eval_set'] = [margin_v]

        model.fit(**fit_kwargs)

        if margin_v is not None:
            preds = model.predict(X_v, base_margin=margin_v)
        else:
            preds = model.predict(X_v)

        return np.sqrt(mean_squared_error(y_v, preds))
    return objective

# Tracking Metrics
results_summary = {}
best_params_dict = {}
best_overall_rmse = float('inf') 
best_overall_exp  = None
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# 5. EXECUTION PHASE 1: Standard & Linear Guess Models
# ==============================================================================
print("\n=== STARTING PHASE 1: STANDARD EXPERIMENTS ===")
for exp_name, cols in standard_experiments.items():
    print(f"Running Optuna for: {exp_name} ({len(cols)} features)")
    
    X_train, y_train = train_df[cols], train_df[target_col]
    X_valid, y_valid = valid_df[cols], valid_df[target_col]
    
    objective_func = create_objective(X_train, y_train, X_valid, y_valid)
    study = optuna.create_study(direction='minimize', study_name=f'XGB_{exp_name}')
    study.optimize(objective_func, n_trials=30) 
    
    results_summary[exp_name] = study.best_value

    best_params_dict[exp_name] = study.best_params
    
    if study.best_value < best_overall_rmse:
        best_overall_rmse = study.best_value
        best_overall_exp  = exp_name


# ==============================================================================
# 7. FINAL SUMMARY
# ==============================================================================
print(f"\n{'='*60}")
print("=== MEGA OPTIMIZATION PIPELINE FINISHED ===")
print("Summary of Best RMSEs across ALL Experiments:")
for exp, rmse_val in results_summary.items():
    print(f"  {exp}: {rmse_val:.4f}")

print(f"\n Winning Feature Set: **{best_overall_exp}** (RMSE: {best_overall_rmse:.4f})")

Loading VAE outputs...
Loading Target and External files...
Filtering dataset...
CRITICAL CHECK - Master DataFrame rows after filtering: 34934

=== STARTING PHASE 1: STANDARD EXPERIMENTS ===
Running Optuna for: Latent BK SIC Only (19 features)


C:\Users\Jonas\miniconda3\envs\mlenv\lib\site-packages\xgboost\core.py:751: UserWarning: [14:31:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Running Optuna for: Latent BK TAS Only (19 features)
Running Optuna for: Latent BK HFLS Only (19 features)
Running Optuna for: Latent BK ZG Only (19 features)
Running Optuna for: Latent BK Region Only (67 features)
Running Optuna for: Latent NAO Region Only (51 features)
Running Optuna for: Latent NP Region Only (515 features)
Running Optuna for: Latent BK + NAO Region Only (115 features)
Running Optuna for: Latent NP + NAO Region Only (563 features)
Running Optuna for: Latent BK + NP Region Only (579 features)
Running Optuna for: Latent SIC Only (147 features)
Running Optuna for: Latent TAS Only (163 features)
Running Optuna for: Latent HFLS Only (163 features)
Running Optuna for: Latent ZG Only (163 features)
Running Optuna for: All Combined Only (627 features)
Running Optuna for: Latent SIC + Linear Guess (152 features)
Running Optuna for: Latent TAS + Linear Guess (168 features)
Running Optuna for: Latent HFLS + Linear Guess (168 features)
Running Optuna for: Latent ZG + Linear Gue

In [3]:
test_pred = {}

def evaluation_plot2(estimator, y_test, y_pred, df_valid, X_valid, exp_name, region='NAO', results_dict=None, margin_valid=None):
    region_titles = {
        'NAO': 'North Atlantic Ocean', 'NP': 'North Pole', 'BK': 'Barents-Kara Sea',
        'BK_NAO': 'Combined BK NAO', 'BK_NP': 'Combined BK NP', 'NAO_NP': 'Combined NAO NP',
        'ALL': 'All Regions Combined'
    }
    full_region_name = region_titles.get(region, region) 
    
    y_test_array = np.array(y_test) 
    residuals = y_test_array - y_pred
    
    with np.errstate(divide='ignore', invalid='ignore'):
        relative_errors = residuals / np.abs(y_test_array)
        
    relative_errors = relative_errors[np.abs(relative_errors) <= 1.5]
    percentile_90 = np.percentile(np.abs(residuals), 90)
    high_residual_mask = np.abs(residuals) <= percentile_90
    
    os.makedirs('Saved_Plots', exist_ok=True)
    safe_exp_name = exp_name.replace(" ", "_").replace("+", "plus")
    
    if "Linear Guess" in exp_name:
        prefix = "lin"
    elif "Warm Start" in exp_name:
        prefix = "ws"
    else:
        prefix = "std"
        
    base_save_path = f"Saved_Plots/vaeb_{prefix}_{region}_{safe_exp_name}" # vaeb for variation beta autoencoder. ae for normal ae

    # ---------------------------------------------------------
    # Plot 1: Error Distributions
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,6))
    fig.suptitle(f'{full_region_name} - {exp_name}: Error Distributions', fontsize=14, fontweight='bold') 
    
    ax1.hist(residuals, bins=100) 
    ax1.set_title('Distribution of Residuals')
    ax1.set_xlabel('Residuals')
    ax1.set_ylabel('Count')
    ax1.set_xlim(-15,15)
    
    ax2.hist(relative_errors, bins=100)
    ax2.set_title('Distribution of Relative Errors')
    ax2.set_xlabel('Relative Prediction Error')
    ax2.set_ylabel('Count')
    ax2.set_xlim(np.min(relative_errors) - 0.05, np.max(relative_errors) + 0.05)
    plt.tight_layout()
    plt.savefig(f'{base_save_path}_Error_Distributions.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    # ---------------------------------------------------------
    # Plot 2: The Diversion
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    sc1 = plt.scatter(y_test_array, y_pred, c='blue', alpha=0.1, label='100th percentile', zorder=2)
    sc2 = plt.scatter(y_test_array[high_residual_mask], y_pred[high_residual_mask], c='green', alpha=0.1, label='residual <= 90th percentile', zorder=3)
    ln1 = plt.plot([min(y_test_array), max(y_test_array)], [min(y_test_array), max(y_test_array)], color='orange', linewidth=2, zorder=1)
    
    model_names = ["r106", "r139", "r117"]
    time_stamp = ["2079-12-16T12:00:00", "2077-01-16T12:00:00", "2069-02-15"]
    time_stamp_read = ["2079-12", "2077-01", "2069-02"]
    
    y_pred_series = pd.Series(y_pred, index=df_valid.index)
    texts, pt_x, pt_y = [], [], []

    for m, t, t_read in zip(model_names, time_stamp, time_stamp_read):
        t_dt = pd.to_datetime(t)
        mask = (df_valid['member'] == m) & (df_valid['time'] == t_dt)
        if mask.any():
            pt_y_test = y_test.loc[mask].values[0] 
            pt_y_pred = y_pred_series.loc[mask].values[0]
            
            plt.scatter(pt_y_test, pt_y_pred, c='red', s=80, zorder=10, edgecolors='black', label='_nolegend_')
            txt = plt.text(pt_y_test, pt_y_pred, f"{m}\n{t_read}", ha='center', va='center', color='darkred', fontweight='bold', zorder=11)
            texts.append(txt)
            pt_x.append(pt_y_test)
            pt_y.append(pt_y_pred)
    
    if results_dict is not None:
        results_dict[exp_name] = list(zip(pt_x, pt_y))
    
    if texts:
        adjust_text(texts, x=pt_x, y=pt_y, arrowprops=dict(arrowstyle='-', color='darkred', lw=1.5, alpha=0.7))

    plt.xlabel('Label')
    plt.ylabel('Prediction')
    plt.title(f'{full_region_name} - {exp_name}: Diversion Scatter', fontsize=14, fontweight='bold') 
    plt.legend()
    plt.savefig(f'{base_save_path}_Diversion.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    # ---------------------------------------------------------
    # Plot 3: RMSE and MAE Evaluation Curves (Logarithmic)
    # ---------------------------------------------------------
    results = estimator.evals_result()
    epochs = len(results['validation_0']['rmse'])
    x_axis = range(0, epochs)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,6))
    fig.suptitle(f'{full_region_name} - {exp_name}: Learning Curves', fontsize=14, fontweight='bold')
    
    ax1.plot(x_axis, results['validation_0']['rmse'], label='Train')
    ax1.plot(x_axis, results['validation_1']['rmse'], label='Validation')
    ax1.legend(loc='upper right')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('RMSE (Log Scale)')
    ax1.set_yscale('log')  
    ax1.set_title('Evolution of the RMSE')
    
    ax2.plot(x_axis, results['validation_0']['mae'], label='Train')
    ax2.plot(x_axis, results['validation_1']['mae'], label='Validation')
    ax2.legend(loc='upper right')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('MAE (Log Scale)')
    ax2.set_yscale('log')  
    ax2.set_title('Evolution of the MAE')
    
    plt.tight_layout()
    plt.savefig(f'{base_save_path}_Learning_Curves.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    # ---------------------------------------------------------
    # Plot 4: SHAP Values Summary
    # ---------------------------------------------------------
    print("    -> Calculating SHAP math...")
    t_shap_math = time.time()
    
    booster = estimator.get_booster()
    booster.set_param({'device': 'cpu'})
    
    dmatrix_kwargs = {'data': X_valid}
    if margin_valid is not None:
        dmatrix_kwargs['base_margin'] = margin_valid
        
    dmatrix = xgb.DMatrix(**dmatrix_kwargs)
    
    shap_contribs = booster.predict(dmatrix, pred_contribs=True, approx_contribs=True)
    shap_values = shap_contribs[:, :-1]
    
    print(f"    -> SHAP Math Done in: {time.time() - t_shap_math:.2f}s")
    print("    -> Drawing SHAP plot...")
    
    plt.figure(figsize=(10, 6))
    plt.title(f'{full_region_name} - {exp_name}: SHAP Values', fontsize=14, fontweight='bold', pad=20)
    shap.summary_plot(shap_values, X_valid, show=False)
    plt.savefig(f'{base_save_path}_SHAP_Summary.png', bbox_inches='tight', dpi=300)
    plt.close('all')


In [6]:
# ==============================================================================
# EXECUTION LOOP: Generate Plots for ALL Experiments
# ==============================================================================
print("\n=== STARTING PHASE 3: GENERATING PLOTS ===")

all_experiments = standard_experiments

for exp_name, cols in all_experiments.items():
    print(f"\n{'='*50}")
    print(f"Processing Plots for: {exp_name}")
    
    t_start = time.time()
    
    if 'BK' in exp_name and 'NAO' in exp_name: region_code = 'BK_NAO'
    elif 'BK' in exp_name and 'NP' in exp_name: region_code = 'BK_NP'
    elif 'NAO' in exp_name and 'NP' in exp_name: region_code = 'NAO_NP'
    elif 'BK' in exp_name: region_code = 'BK'
    elif 'NAO' in exp_name: region_code = 'NAO'
    elif 'NP' in exp_name: region_code = 'NP'
    else: region_code = 'ALL'

    X_train_plot = train_df[cols]
    y_train_plot = train_df[target_col]
    X_valid_plot = valid_df[cols]
    y_valid_plot = valid_df[target_col]
    
    best_params = best_params_dict.get(exp_name)
    if best_params is None:
        print(f"Skipping {exp_name}: No optimized parameters found.")
        continue
        
    plot_model = xgb.XGBRegressor(
        **best_params, 
        random_state=42, 
        tree_method='hist',   
        device='cuda',            
        objective='reg:squarederror', 
        eval_metric=['mae', 'rmse'],
        early_stopping_rounds=30
    )
    
    fit_kwargs = {
        'X': X_train_plot, 'y': y_train_plot,
        'eval_set': [(X_train_plot, y_train_plot), (X_valid_plot, y_valid_plot)],
        'verbose': False
    }
    

    plot_model.fit(**fit_kwargs)

    preds_plot = plot_model.predict(X_valid_plot)
    margin_for_shap = None
        
    t_train = time.time()
    print(f"  -> Model Retrained & Predicted: {t_train - t_start:.2f} seconds")
    
    evaluation_plot2(
        plot_model, y_valid_plot, preds_plot, valid_df, X_valid_plot, 
        exp_name, region=region_code, results_dict=test_pred, 
        margin_valid=margin_for_shap
    )
    
    t_plot = time.time()
    print(f"  -> Plots Generated:             {t_plot - t_train:.2f} seconds")
    
    del plot_model, preds_plot, X_train_plot, y_train_plot, X_valid_plot, y_valid_plot
    gc.collect()

print("\nAll pipelines completed successfully. Check the 'Saved_Plots' directory!")


=== STARTING PHASE 3: GENERATING PLOTS ===

Processing Plots for: Latent BK SIC Only
  -> Model Retrained & Predicted: 6.72 seconds
    -> Calculating SHAP math...
    -> SHAP Math Done in: 0.04s
    -> Drawing SHAP plot...
  -> Plots Generated:             3.55 seconds

Processing Plots for: Latent BK TAS Only
  -> Model Retrained & Predicted: 8.98 seconds
    -> Calculating SHAP math...
    -> SHAP Math Done in: 0.12s
    -> Drawing SHAP plot...
  -> Plots Generated:             3.31 seconds

Processing Plots for: Latent BK HFLS Only
  -> Model Retrained & Predicted: 0.91 seconds
    -> Calculating SHAP math...
    -> SHAP Math Done in: 0.01s
    -> Drawing SHAP plot...
  -> Plots Generated:             3.20 seconds

Processing Plots for: Latent BK ZG Only
  -> Model Retrained & Predicted: 1.32 seconds
    -> Calculating SHAP math...
    -> SHAP Math Done in: 0.01s
    -> Drawing SHAP plot...
  -> Plots Generated:             3.35 seconds

Processing Plots for: Latent BK Region Only

In [7]:
print("\n=== GENERATING LATEX TABLES ===")

def generate_latex_table(experiment_keys, caption, label):
    latex_str = "\\begin{table}[htbp]\n"
    latex_str += "\\centering\n"
    latex_str += "\\begin{tabular}{l c c c c}\n"
    latex_str += "\\toprule\n"
    latex_str += "\\textbf{Experiment Name} & \\textbf{RMSE} & \\textbf{r106 Diff} & \\textbf{r139 Diff} & \\textbf{r117 Diff} \\\\\n"
    latex_str += "\\midrule\n"

    for exp in experiment_keys:
        if exp in results_summary and exp in test_pred:
            rmse_val = results_summary[exp]
            
            diff_r106 = test_pred[exp][0][0] - test_pred[exp][0][1]
            diff_r139 = test_pred[exp][1][0] - test_pred[exp][1][1]
            diff_r117 = test_pred[exp][2][0] - test_pred[exp][2][1]
            
            safe_exp = exp.replace("_", "\\_").replace("+", "$+$")
            
            latex_str += f"{safe_exp} & {rmse_val:.2f} & {diff_r106:.2f} & {diff_r139:.2f} & {diff_r117:.2f} \\\\\n"

    latex_str += "\\bottomrule\n"
    latex_str += "\\end{tabular}\n"
    latex_str += f"\\caption{{{caption}}}\n"
    latex_str += f"\\label{{{label}}}\n"
    latex_str += "\\end{table}\n"
    
    return latex_str

linear_exps = [exp for exp in results_summary.keys() if 'Linear Guess' in exp]
standard_exps = [exp for exp in results_summary.keys() if 'Only' in exp]
warm_start_exps = [exp for exp in results_summary.keys() if 'Warm Start' in exp]

table_linear = generate_latex_table(
    linear_exps, 
    "Results Summary and Model Differences ($y_{test} - y_{pred}$) - Linear Guesses", 
    "tab:results_diff_linear"
)

table_standard = generate_latex_table(
    standard_exps, 
    "Results Summary and Model Differences ($y_{test} - y_{pred}$) - Standard Latent Spaces", 
    "tab:results_diff_standard"
)

table_ws = generate_latex_table(
    warm_start_exps, 
    "Results Summary and Model Differences ($y_{test} - y_{pred}$) - Warm Starts", 
    "tab:results_diff_ws"
)

print(table_linear)
print("\n" + "% " + "="*50 + "\n")
print(table_standard)
print("\n" + "% " + "="*50 + "\n")
print(table_ws)


=== GENERATING LATEX TABLES ===
\begin{table}[htbp]
\centering
\begin{tabular}{l c c c c}
\toprule
\textbf{Experiment Name} & \textbf{RMSE} & \textbf{r106 Diff} & \textbf{r139 Diff} & \textbf{r117 Diff} \\
\midrule
Latent SIC $+$ Linear Guess & 2.56 & -8.76 & -10.18 & -9.13 \\
Latent TAS $+$ Linear Guess & 2.27 & -5.85 & -8.54 & -4.86 \\
Latent HFLS $+$ Linear Guess & 2.39 & -9.68 & -10.03 & -6.70 \\
Latent ZG $+$ Linear Guess & 2.12 & -8.65 & -7.36 & -6.10 \\
Latent BK Region $+$ Linear Guess & 2.35 & -7.86 & -7.14 & -8.25 \\
All Combined $+$ Linear Guess & 2.01 & -7.42 & -6.39 & -6.20 \\
\bottomrule
\end{tabular}
\caption{Results Summary and Model Differences ($y_{test} - y_{pred}$) - Linear Guesses}
\label{tab:results_diff_linear}
\end{table}


% ==================================================

\begin{table}[htbp]
\centering
\begin{tabular}{l c c c c}
\toprule
\textbf{Experiment Name} & \textbf{RMSE} & \textbf{r106 Diff} & \textbf{r139 Diff} & \textbf{r117 Diff} \\
\midrule
Late